# MultiTaxi plain DQN baseline

Run from `app/`. This notebook uses only `configs/dqn.yaml`; Reward Machine variants belong in separate experiments.

Use `BenchmarkConfiguration(config_file='dqn.yaml')` for one passenger or `BenchmarkConfiguration(config_file='dqn_2p.yaml')` for two passengers.

In [ ]:
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

from scripts import train_dqn_agent
from src.config import Configuration
from src.models import evaluate_agent


In [ ]:
@dataclass
class BenchmarkConfiguration:
    training_seeds: tuple[int, ...] = tuple(range(42, 52))
    convergence_interval: int = 1000
    num_passengers: int | None = None
    n_training_episodes: int | None = None
    n_eval_episodes: int = 100
    eval_seed_base: int = 2
    config_file: str = 'dqn.yaml'
    record_version: str = 'plain-dqn-v1'
    rerun_completed: bool = False
    DATA_PATH: str = '../data'
    VIDEO_PATH: str = '../videos'

    @property
    def results_path(self):
        config = Configuration(yaml_config_path=self.config_file)
        grid = f'{config.multitaxi_grid_size}x{config.multitaxi_grid_size}'
        passengers = config.multitaxi_num_passengers if self.num_passengers is None else self.num_passengers
        return Path(self.DATA_PATH, f'multitaxi_{grid}_{passengers}p_{self.record_version}.json')

    def evaluation_episodes(self, n_training_episodes):
        episodes = list(range(self.convergence_interval, n_training_episodes + 1, self.convergence_interval))
        if not episodes or episodes[-1] != n_training_episodes:
            episodes.append(n_training_episodes)
        return tuple(episodes)

    def config_for(self, seed):
        config = Configuration(yaml_config_path=self.config_file)
        if self.num_passengers is not None:
            config.multitaxi_num_passengers = self.num_passengers
        if self.n_training_episodes is not None:
            config.n_training_episodes = self.n_training_episodes
        config.exp_name = f'dqn_{config.multitaxi_num_passengers}p'
        config.n_eval_episodes = self.n_eval_episodes
        config.eval_seed_base = self.eval_seed_base + self.training_seeds.index(seed)
        config.VIDEO_PATH = self.VIDEO_PATH
        config.set_seed(seed)
        return config


def resolved_training_config(config):
    resolved = asdict(config)
    for name in ('CONFIGS_PATH', 'DATA_PATH', 'MODELS_PATH', 'LOGS_PATH', 'VIDEO_PATH', 'seed', 'eval_seed', 'eval_seed_base'):
        resolved.pop(name)
    return resolved


BENCHMARK = BenchmarkConfiguration()


def benchmark_spec():
    reference_config = BENCHMARK.config_for(BENCHMARK.training_seeds[0])
    return {
        'version': 1,
        'algorithm': 'plain_dqn',
        'config_file': BENCHMARK.config_file,
        'config_yaml': Path(reference_config.CONFIGS_PATH, BENCHMARK.config_file).read_text(encoding='utf-8'),
        'resolved_config': resolved_training_config(reference_config),
        'training_seeds': list(BENCHMARK.training_seeds),
        'evaluation_episodes': BENCHMARK.n_eval_episodes,
        'evaluation_seed_base': BENCHMARK.eval_seed_base,
    }


REFERENCE_CONFIG = BENCHMARK.config_for(BENCHMARK.training_seeds[0])
BENCHMARK_SPEC = benchmark_spec()
RUN_TRAINING = False
print(f'Runs: {len(BENCHMARK.training_seeds)}')
print(f'Training episodes per run: {REFERENCE_CONFIG.n_training_episodes}')
print(f'Passengers: {REFERENCE_CONFIG.multitaxi_num_passengers}')
print(f'Results: {BENCHMARK.results_path}')


In [ ]:
def normalize_metrics(metrics):
    normalized = {}
    for key, value in metrics.items():
        if isinstance(value, np.generic):
            value = value.item()
        normalized[key] = None if isinstance(value, float) and not np.isfinite(value) else value
    return normalized


def load_runs():
    if not BENCHMARK.results_path.exists():
        return []
    document = json.loads(BENCHMARK.results_path.read_text(encoding='utf-8'))
    if not isinstance(document, dict) or document.get('spec') != benchmark_spec():
        raise ValueError('Saved results use a different benchmark specification')
    runs = document.get('runs')
    if not isinstance(runs, list):
        raise ValueError('Experiment runs must be a list')
    return runs


def save_runs(runs):
    BENCHMARK.results_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = BENCHMARK.results_path.with_suffix(f'{BENCHMARK.results_path.suffix}.tmp')
    temporary_path.write_text(
        json.dumps({'spec': benchmark_spec(), 'runs': runs}, indent=2, allow_nan=False),
        encoding='utf-8',
    )
    temporary_path.replace(BENCHMARK.results_path)


def completed_run(run):
    return run['metrics'] is not None and {
        checkpoint['episode'] for checkpoint in run['convergence']
    } == set(BENCHMARK.evaluation_episodes(run['training_episodes']))


def completed_runs():
    runs = [run for run in load_runs() if completed_run(run)]
    completed_seeds = {run['seed'] for run in runs}
    missing = [str(seed) for seed in BENCHMARK.training_seeds if seed not in completed_seeds]
    if missing:
        raise ValueError(f"Benchmark is incomplete for seeds: {', '.join(missing)}")
    return runs


In [ ]:
if RUN_TRAINING:
    runs = load_runs()
    for seed in BENCHMARK.training_seeds:
        existing_run = next((run for run in runs if run['seed'] == seed), None)
        config = BENCHMARK.config_for(seed)
        grid = f'{config.multitaxi_grid_size}x{config.multitaxi_grid_size}'
        video_path = Path(config.VIDEO_PATH, f'{grid}_{config.exp_name}_seed{seed}_video.gif')
        if existing_run and completed_run(existing_run) and video_path.is_file() and not BENCHMARK.rerun_completed:
            print(f'Skipping seed {seed}')
            continue

        progress_bar = tqdm(total=config.n_training_episodes, desc=f'DQN | seed {seed}', unit='episode')
        run = {
            'seed': seed,
            'training_episodes': config.n_training_episodes,
            'evaluation_seeds': config.eval_seed,
            'metrics': None,
            'convergence': [],
        }
        runs = [stored_run for stored_run in runs if stored_run['seed'] != seed]
        runs.append(run)
        save_runs(runs)

        def progress_callback(episode, agent, env, get_propositions):
            progress_bar.update(episode - progress_bar.n)
            if episode not in BENCHMARK.evaluation_episodes(config.n_training_episodes):
                return
            metrics = normalize_metrics(evaluate_agent(
                config, agent, get_propositions, env,
                seeds=run['evaluation_seeds'], report=False, return_metrics=True,
            ))
            run['convergence'].append({'episode': episode, 'metrics': metrics})
            save_runs(runs)
            tqdm.write(
                f"DQN, seed {seed}, episode {episode}/{config.n_training_episodes}: "
                f"reward={metrics['mean_reward']:.2f}, success={metrics['successes']}/{metrics['episodes']}"
            )

        try:
            run['metrics'] = normalize_metrics(train_dqn_agent(config, progress_callback=progress_callback))
        finally:
            progress_bar.close()
        if not completed_run(run):
            raise RuntimeError('Training finished without all convergence evaluations')
        save_runs(runs)


In [ ]:
def summarize_results(runs):
    successes = [run['metrics']['successes'] / run['metrics']['episodes'] for run in runs]
    rewards = np.asarray([run['metrics']['mean_reward'] for run in runs])
    steps = [run['metrics']['mean_successful_steps'] for run in runs if run['metrics']['mean_successful_steps'] is not None]
    display({
        'runs': len(runs),
        'success_rate': float(np.mean(successes)),
        'mean_reward': float(rewards.mean()),
        'reward_std_across_runs': float(rewards.std()),
        'mean_successful_steps': float(np.mean(steps)) if steps else None,
    })


In [ ]:
def plot_final_evaluation(runs):
    figure = make_subplots(rows=1, cols=2, subplot_titles=('Final reward', 'Successful steps'))
    figure.add_trace(go.Box(y=[run['metrics']['mean_reward'] for run in runs], name='DQN', boxpoints='all'), row=1, col=1)
    steps = [run['metrics']['mean_successful_steps'] for run in runs if run['metrics']['mean_successful_steps'] is not None]
    if steps:
        figure.add_trace(go.Box(y=steps, name='DQN', boxpoints='all', showlegend=False), row=1, col=2)
    figure.update_layout(title='MultiTaxi plain DQN final evaluation', template='plotly_white', height=550)
    figure.show()


In [ ]:
def plot_convergence(runs):
    values_by_episode = {}
    for run in runs:
        for checkpoint in run['convergence']:
            values_by_episode.setdefault(checkpoint['episode'], []).append(checkpoint['metrics']['successes'] / checkpoint['metrics']['episodes'])
    episodes = sorted(values_by_episode)
    values = [values_by_episode[episode] for episode in episodes]
    figure = go.Figure(go.Scatter(
        x=episodes, y=[np.mean(items) for items in values],
        error_y={'type': 'data', 'array': [np.std(items) for items in values], 'visible': True},
        mode='lines+markers', name='DQN',
    ))
    figure.update_layout(title='Plain DQN success convergence', xaxis_title='Training episodes', yaxis_title='Success rate', template='plotly_white', height=600)
    figure.show()


## Plot saved results

Set `RUN_TRAINING = True` only for a fresh run whose specification matches the saved result document.

In [ ]:
runs = completed_runs()
summarize_results(runs)
plot_final_evaluation(runs)
plot_convergence(runs)
